In [1]:
import os
import pandas as pd

In [2]:
import numpy as np
from sklearn.metrics import accuracy_score, classification_report, balanced_accuracy_score
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.tree import DecisionTreeClassifier
## PCA
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from imblearn.over_sampling import SMOTE

## svm
from sklearn.svm import SVC
import pandas as pd


In [3]:
ML_CLASSIFIER = ['MLP', 'GaussianNB', "Adaboost", "KNN", "RFClassifier", "SVM_linear", "SVM_sigmoid", "SVM_RBF", "XGBoost"] # "ELM"

def classify(X_train, y_train, X_test, y_test, classifier, search_type='grid'):
    """
    Classify the data using a Random Forest Classifier
    :param X_train: Training data
    :param y_train: Training labels
    :param X_test: Testing data
    :param y_test: Testing labels
    :param search_type: Type of search to perform
    :return: Accuracy of the classifier
    """
    # Create a Random Forest Classifier
    #clf = RandomForestClassifier()

    if classifier == "MLP":
        # clf = MLPClassifier(hidden_layer_sizes=(100,100, 50), max_iter=1000) ### uncomment if the performance is not good with below hyperparameter
        clf = MLPClassifier(hidden_layer_sizes=(100,100, 50), max_iter=1000)
    elif classifier == "GaussianNB": ### {'priors': None, 'var_smoothing': 1e-05}
        clf = GaussianNB(priors=None, var_smoothing= 1e-05) ### Often, the default parameters work well for many datasets.
    elif classifier == "Adaboost":
        dtc = DecisionTreeClassifier(ccp_alpha=0.0, class_weight='balanced', criterion='entropy', max_depth=10, max_features=None, max_leaf_nodes=None, min_impurity_decrease=0.0, min_samples_leaf=2, min_samples_split=2, splitter='best')
        clf = AdaBoostClassifier(estimator=dtc, n_estimators=200, random_state=0, learning_rate=1) ###{'learning_rate': 1, 'n_estimators': 200}
        
    elif classifier == "KNN":
        clf = KNeighborsClassifier(algorithm='auto', leaf_size=10, metric='euclidean', n_jobs=-1,  n_neighbors=1, p=1, weights='uniform')  
    elif classifier == "RFClassifier": ### 
        clf = RandomForestClassifier(bootstrap = False, criterion = 'entropy', max_depth = None, max_features = 'sqrt', min_samples_leaf = 1, min_samples_split = 2, n_estimators = 100, oob_score = False, random_state = 42)
    elif classifier == "SVM_linear": ##
        clf = SVC(kernel='linear')
    elif classifier == "SVM_sigmoid": ###
        clf = SVC(kernel='sigmoid')
    elif classifier == "SVM_RBF": 
        clf = SVC(C=10, cache_size = 200, class_weight = None, gamma = 'scale', kernel = 'rbf', max_iter = -1, probability = True, shrinking = True, tol = 0.001)
    elif classifier == "XGBoost": #'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 300, 'subsample': 0.7
        clf = XGBClassifier(n_estimators=200, learning_rate=0.1, max_depth=3,subsample=0.7)
    else:
        print("Invalid classifier")
        return None
        # Fit the model
    clf.fit(X_train, y_train)

    # Predict the test data
    y_pred = clf.predict(X_test)

    # Calculate the accuracy
    accuracy = balanced_accuracy_score(y_test, y_pred)
    
    return accuracy


In [9]:
data = pd.read_csv('BT-small-2c-dataset_results_finetune_ALL_Models.csv')
data.head(15)

,Model,XGBoost,MLP,GaussianNB,Adaboost,KNN,RFClassifier,SVM_linear,SVM_sigmoid,SVM_RBF
0,resnet50,0.737097,0.728226,0.597581,0.600000,0.738710,0.667742,0.728226,0.420968,0.817742
1,resnet101,0.644355,0.678226,0.787097,0.625000,0.740323,0.651613,0.662097,0.513710,0.685484
2,densenet121,0.808871,0.867742,0.772581,0.750000,0.738710,0.825000,0.867742,0.851613,0.842742
3,densenet169,0.875000,0.867742,0.812097,0.775000,0.803226,0.850000,0.835484,0.794355,0.858871
4,vgg16,0.958871,0.908871,0.735484,0.850000,0.804839,0.875000,0.851613,0.908871,0.892742
5,vgg19,0.851613,0.860484,0.769355,0.858871,0.845968,0.776613,0.835484,0.851613,0.885484
6,alexnet,0.867742,0.925000,0.745968,0.850000,0.772581,0.883871,0.933871,0.758871,0.950000
7,resnext50_32x4d,0.637097,0.794355,0.631452,0.625000,0.606452,0.633871,0.753226,0.500000,0.751613
8,resnext101_32x8d,0.783871,0.851613,0.722581,0.625000,0.729839,0.742742,0.826613,0.653226,0.833871
9,shufflenet_v2_x1_0,0.785484,0.908871,0.658871,0.725000,0.763710,0.751613,0.908871,0.900000,0.900000


In [10]:
## taking row-wise average of all the data 
data['average'] = data.drop("Model", axis=1).mean(axis=1)


In [11]:
data.drop("Model", axis=1).mean(axis=0)

XGBoost         0.859968
MLP             0.901194
GaussianNB      0.782935
Adaboost        0.825129
KNN             0.813548
RFClassifier    0.845806
SVM_linear      0.878645
SVM_sigmoid     0.835452
SVM_RBF         0.895290
average         0.848663
dtype: float64

In [7]:
## add coumn-wise average of all the data
#data.loc['average'] = data.drop("Model", axis=1).mean(axis=0)

In [12]:
data

,Model,XGBoost,MLP,GaussianNB,Adaboost,KNN,RFClassifier,SVM_linear,SVM_sigmoid,SVM_RBF,average
0,resnet50,0.737097,0.728226,0.597581,0.600000,0.738710,0.667742,0.728226,0.420968,0.817742,0.670699
1,resnet101,0.644355,0.678226,0.787097,0.625000,0.740323,0.651613,0.662097,0.513710,0.685484,0.665323
2,densenet121,0.808871,0.867742,0.772581,0.750000,0.738710,0.825000,0.867742,0.851613,0.842742,0.813889
3,densenet169,0.875000,0.867742,0.812097,0.775000,0.803226,0.850000,0.835484,0.794355,0.858871,0.830197
4,vgg16,0.958871,0.908871,0.735484,0.850000,0.804839,0.875000,0.851613,0.908871,0.892742,0.865143
5,vgg19,0.851613,0.860484,0.769355,0.858871,0.845968,0.776613,0.835484,0.851613,0.885484,0.837276
6,alexnet,0.867742,0.925000,0.745968,0.850000,0.772581,0.883871,0.933871,0.758871,0.950000,0.854211
7,resnext50_32x4d,0.637097,0.794355,0.631452,0.625000,0.606452,0.633871,0.753226,0.500000,0.751613,0.659229
8,resnext101_32x8d,0.783871,0.851613,0.722581,0.625000,0.729839,0.742742,0.826613,0.653226,0.833871,0.752151
9,shufflenet_v2_x1_0,0.785484,0.908871,0.658871,0.725000,0.763710,0.751613,0.908871,0.900000,0.900000,0.811380


In [13]:
## sort the data by average
data = data.sort_values(by='average', ascending=False)
data

,Model,XGBoost,MLP,GaussianNB,Adaboost,KNN,RFClassifier,SVM_linear,SVM_sigmoid,SVM_RBF,average
19,vit_small_patch16_224,1.000000,0.975000,0.828226,0.975000,0.901613,0.950000,0.933871,0.975000,0.975000,0.945968
12,vit_base_patch16_224,1.000000,0.975000,0.894355,0.925000,0.910484,0.925000,0.975000,0.933871,0.975000,0.945968
15,vit_small_patch32_224,0.942742,0.950000,0.910484,0.958871,0.910484,0.933871,0.950000,0.958871,0.975000,0.943369
20,vit_base_patch16_384,0.933871,0.958871,0.903226,0.975000,0.885484,0.950000,0.975000,0.933871,0.958871,0.941577
18,vit_tiny_patch16_224,0.975000,0.975000,0.894355,0.950000,0.903226,0.925000,0.851613,0.942742,0.975000,0.932437
22,vit_small_patch32_384,0.867742,0.958871,0.908871,0.925000,0.942742,0.933871,0.935484,0.933871,0.926613,0.925896
24,vit_base_patch32_384,0.900000,0.975000,0.813710,0.925000,0.901613,0.925000,0.975000,0.933871,0.975000,0.924910
13,vit_base_patch32_224,0.933871,0.950000,0.845968,0.908871,0.935484,0.958871,0.901613,0.933871,0.933871,0.922491
17,vit_base_patch8_224,0.925000,0.950000,0.878226,0.925000,0.885484,0.900000,0.933871,0.933871,0.950000,0.920161
23,vit_small_patch16_384,0.950000,0.933871,0.876613,0.925000,0.812097,0.917742,0.917742,0.958871,0.975000,0.918548


In [14]:
list(data.head(3)['Model'])

['vit_small_patch16_224', 'vit_base_patch16_224', 'vit_small_patch32_224']

### Top 3 performance model

In [15]:
top_model_combinations = [
                          ['vit_small_patch16_224', 'vit_base_patch16_224'],
                          ['vit_small_patch32_224', 'vit_small_patch16_224'],
                          ['vit_base_patch16_224', 'vit_small_patch32_224'],
                          ['vit_small_patch16_224', 'vit_base_patch16_224', 'vit_small_patch32_224']
                          ]

Ensemble vgg16 with Top 2 networks ['vit_base_patch32_384', 'vit_small_patch32_224', 'vgg16']

In [12]:



# top_model_combinations = [
#                           ['vit_base_patch32_384', 'vit_small_patch32_224'],
#                           ['vit_base_patch32_384', 'vgg16'],
#                           ['vit_small_patch32_224', 'vgg16'],
#                           ['vit_base_patch32_384', 'vit_small_patch32_224', 'vgg16']
#                           ]

In [22]:
columns = ['Model', "XGBoost", 'MLP', 'GaussianNB', "Adaboost", "KNN", "RFClassifier", "SVM_linear", "SVM_sigmoid", "SVM_RBF", ]

dataframe = pd.DataFrame(columns=columns)
# add 12 rows to the dataframe with zero values 
for model_list in top_model_combinations:
    model_name = ' + '.join(model_list)
    new_row = {'Model': model_name, "XGBoost":0, 'MLP': 0, 'GaussianNB': 0, "Adaboost": 0, "KNN": 0, "RFClassifier": 0, "SVM_linear": 0, "SVM_sigmoid": 0, "SVM_RBF": 0} 
    dataframe.loc[len(dataframe)] = new_row

model_versions = ['simple_version', "with_normalization_&_PCA", "with_SMOTE_only", "with_normalization_&_PCA_SMOTE"]

model_version_index = 3
main_path = 'extracted_features_BT-small-2c'
for ml_classifier in ML_CLASSIFIER:
    for model_list in top_model_combinations:
        print('Model List:', model_list)
        ensemble_X_train = []
        ensemble_X_test = []

        for model in model_list:
            sub_dir = os.path.join(main_path, model)
            # Load the data
            X_train = np.load(os.path.join(sub_dir, 'train_data_array_features.npy'))
            y_train = np.load(os.path.join(sub_dir, 'train_data_array_labels.npy'))
            X_test = np.load(os.path.join(sub_dir, 'test_data_array_features.npy'))
            y_test = np.load(os.path.join(sub_dir, 'test_data_array_labels.npy'))


            ## squeeze the dimensions 1 from the features 
            X_train = np.squeeze(X_train, axis=1)
            X_test = np.squeeze(X_test, axis=1)

            ## concatenate the features on axis 1
            ensemble_X_train.append(X_train)
            ensemble_X_test.append(X_test)
            

        X_train = np.concatenate(ensemble_X_train, axis=1)
        y_train = y_train
        X_test = np.concatenate(ensemble_X_test, axis=1)
        y_test = y_test

        # apply the standard scaler and PCA
        scaler = StandardScaler()
        X_train_normalized = scaler.fit_transform(X_train)
        X_test_normalized = scaler.transform(X_test)

        # Step 2 & 3: Apply PCA
        n_components = int(0.45 * X_train.shape[1])
        pca = PCA(n_components=n_components)
        X_train = pca.fit_transform(X_train_normalized)
        X_test = pca.transform(X_test_normalized)

        ## no of example in each class
        print("no of example in each class before SMOTE:", np.bincount(y_train))


        # ## over sample the data (data augmentation)
        smote = SMOTE(sampling_strategy='minority')
        X_train, y_train = smote.fit_resample(X_train, y_train)

        ## no of example in each class
        print("no of example in each class after SMOTE:", np.bincount(y_train))


        # print("no of features in X_train:", X_train.shape)

        
        # Classify the data
        accuracy = classify(X_train, y_train, X_test, y_test, ml_classifier)
        print('Accuracy:', accuracy)
        model_name = ' + '.join(model_list)

        dataframe.loc[dataframe['Model'] == model_name, ml_classifier] = accuracy

    print(dataframe)
    dataframe.to_csv(f'BT-small-2c-dataset_results_top_three_{model_versions[model_version_index]}.csv', index=False)


Model List: ['vit_small_patch16_224', 'vit_base_patch16_224']
no of example in each class before SMOTE: [266 439]
no of example in each class after SMOTE: [439 439]
Accuracy: 0.975
Model List: ['vit_small_patch32_224', 'vit_small_patch16_224']


C:\Users\user\AppData\Local\Temp\ipykernel_8708\2067079457.py:74: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.975' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  dataframe.loc[dataframe['Model'] == model_name, ml_classifier] = accuracy


no of example in each class before SMOTE: [266 439]
no of example in each class after SMOTE: [439 439]
Accuracy: 1.0
Model List: ['vit_base_patch16_224', 'vit_small_patch32_224']
no of example in each class before SMOTE: [266 439]
no of example in each class after SMOTE: [439 439]
Accuracy: 0.9588709677419355
Model List: ['vit_small_patch16_224', 'vit_base_patch16_224', 'vit_small_patch32_224']
no of example in each class before SMOTE: [266 439]
no of example in each class after SMOTE: [439 439]
Accuracy: 0.9588709677419355
                                               Model  XGBoost       MLP  \
0       vit_small_patch16_224 + vit_base_patch16_224        0  0.975000   
1      vit_small_patch32_224 + vit_small_patch16_224        0  1.000000   
2       vit_base_patch16_224 + vit_small_patch32_224        0  0.958871   
3  vit_small_patch16_224 + vit_base_patch16_224 +...        0  0.958871   

   GaussianNB  Adaboost  KNN  RFClassifier  SVM_linear  SVM_sigmoid  SVM_RBF  
0           0  

C:\Users\user\AppData\Local\Temp\ipykernel_8708\2067079457.py:74: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.6177419354838709' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  dataframe.loc[dataframe['Model'] == model_name, ml_classifier] = accuracy


no of example in each class before SMOTE: [266 439]
no of example in each class after SMOTE: [439 439]
Accuracy: 0.635483870967742
Model List: ['vit_small_patch16_224', 'vit_base_patch16_224', 'vit_small_patch32_224']
no of example in each class before SMOTE: [266 439]
no of example in each class after SMOTE: [439 439]
Accuracy: 0.5177419354838709
                                               Model  XGBoost       MLP  \
0       vit_small_patch16_224 + vit_base_patch16_224        0  0.975000   
1      vit_small_patch32_224 + vit_small_patch16_224        0  1.000000   
2       vit_base_patch16_224 + vit_small_patch32_224        0  0.958871   
3  vit_small_patch16_224 + vit_base_patch16_224 +...        0  0.958871   

   GaussianNB  Adaboost  KNN  RFClassifier  SVM_linear  SVM_sigmoid  SVM_RBF  
0    0.617742         0    0             0           0            0        0  
1    0.601613         0    0             0           0            0        0  
2    0.635484         0    0         

...

no of example in each class before SMOTE: [266 439]
no of example in each class after SMOTE: [439 439]
Accuracy: 0.9427419354838709
Model List: ['vit_small_patch16_224', 'vit_base_patch16_224', 'vit_small_patch32_224']
no of example in each class before SMOTE: [266 439]
no of example in each class after SMOTE: [439 439]
Accuracy: 0.9427419354838709
                                               Model  XGBoost       MLP  \
0       vit_small_patch16_224 + vit_base_patch16_224        0  0.975000   
1      vit_small_patch32_224 + vit_small_patch16_224        0  1.000000   
2       vit_base_patch16_224 + vit_small_patch32_224        0  0.958871   
3  vit_small_patch16_224 + vit_base_patch16_224 +...        0  0.958871   

   GaussianNB  Adaboost       KNN  RFClassifier  SVM_linear  SVM_sigmoid  \
0    0.617742  0.883871  0.903226      0.858871       0.975     0.975000   
1    0.601613  0.917742  0.870968      0.883871       0.975     0.942742   
2    0.635484  0.983871  0.845968      0.9000

C:\Users\user\AppData\Local\Temp\ipykernel_8708\2067079457.py:74: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.975' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  dataframe.loc[dataframe['Model'] == model_name, ml_classifier] = accuracy


Accuracy: 0.975
Model List: ['vit_base_patch16_224', 'vit_small_patch32_224']
no of example in each class before SMOTE: [266 439]
no of example in each class after SMOTE: [439 439]
Accuracy: 0.975
Model List: ['vit_small_patch16_224', 'vit_base_patch16_224', 'vit_small_patch32_224']
no of example in each class before SMOTE: [266 439]
no of example in each class after SMOTE: [439 439]
Accuracy: 0.975
                                               Model  XGBoost       MLP  \
0       vit_small_patch16_224 + vit_base_patch16_224        0  0.975000   
1      vit_small_patch32_224 + vit_small_patch16_224        0  1.000000   
2       vit_base_patch16_224 + vit_small_patch32_224        0  0.958871   
3  vit_small_patch16_224 + vit_base_patch16_224 +...        0  0.958871   

   GaussianNB  Adaboost       KNN  RFClassifier  SVM_linear  SVM_sigmoid  \
0    0.617742  0.883871  0.903226      0.858871       0.975     0.975000   
1    0.601613  0.917742  0.870968      0.883871       0.975     0.942

C:\Users\user\AppData\Local\Temp\ipykernel_8708\2067079457.py:74: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.9338709677419355' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  dataframe.loc[dataframe['Model'] == model_name, ml_classifier] = accuracy


Accuracy: 0.9104838709677419
Model List: ['vit_base_patch16_224', 'vit_small_patch32_224']
no of example in each class before SMOTE: [266 439]
no of example in each class after SMOTE: [439 439]
Accuracy: 0.9588709677419355
Model List: ['vit_small_patch16_224', 'vit_base_patch16_224', 'vit_small_patch32_224']
no of example in each class before SMOTE: [266 439]
no of example in each class after SMOTE: [439 439]
Accuracy: 0.9588709677419355
                                               Model   XGBoost       MLP  \
0       vit_small_patch16_224 + vit_base_patch16_224  0.933871  0.975000   
1      vit_small_patch32_224 + vit_small_patch16_224  0.910484  1.000000   
2       vit_base_patch16_224 + vit_small_patch32_224  0.958871  0.958871   
3  vit_small_patch16_224 + vit_base_patch16_224 +...  0.958871  0.958871   

   GaussianNB  Adaboost       KNN  RFClassifier  SVM_linear  SVM_sigmoid  \
0    0.617742  0.883871  0.903226      0.858871       0.975     0.975000   
1    0.601613  0.917742  

In [19]:
n_components

768

In [14]:
dataframe

,Model,XGBoost,MLP,GaussianNB,Adaboost,KNN,RFClassifier,SVM_linear,SVM_sigmoid,SVM_RBF
0,vit_base_patch32_384 + vit_small_patch32_224,0.736086,0.762759,0.334131,0.731610,0.771137,0.738784,0.799189,0.683773,0.754797
1,vit_base_patch32_384 + vit_small_patch16_224,0.733586,0.758041,0.333893,0.754932,0.796542,0.748919,0.779797,0.658766,0.754797
2,vit_small_patch32_224 + vit_small_patch16_224,0.740533,0.756554,0.410204,0.764571,0.812421,0.777488,0.719953,0.619690,0.765811
3,vit_base_patch32_384 + vit_small_patch32_224 +...,0.721848,0.767432,0.325231,0.747432,0.802421,0.782297,0.771554,0.665940,0.762297
